In [1]:
# thermo module
# table of contents:

# 0.....imports&data ingestion
# 1.....existing diesel generator (ourDGEN)
# 2.....new auxiliary boilers&generators (nuBG)
# 3.....new generators
# 3a..........methanol generator
# 4.....canned clean coal project
# 5.....existing power plant framework

In [ ]:
# 0
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re 

def load_fbx_weather_data(
    annual_csv_path: str = '/workspaces/uaf-blackstart-tools/data/FBX_annual_temp.csv',
    monthly_txt_path: str = '/workspaces/uaf-blackstart-tools/data/2022-2026 FBX Weather.txt'
) -> dict:
    """
    load Fairbanks weather data from CSV and NOAA F-6 text reports,
    compute and return average monthly temperatures as 'month_temp'.

    returns
    -------
    dict
        month_temp -- mapping of month name (str) to average temperature in degF (float).
        e.g. month_temp['JANUARY'] -> 3.1

    sources
    ------------
    - FBX_annual_temp.csv: Alaska Climate Research Center (1929-2022 annual temps)
    - 2022-2026 FBX Weather.txt: NOAA/NWS Fairbanks F-6 climatological reports
    """

    df_annual = pd.read_csv(annual_csv_path, skiprows=5)

    with open(monthly_txt_path, 'r') as f:
        text = f.read()

    # split on report boundary: each report starts with "PRELIMINARY LOCAL CLIMATOLOGICAL DATA"
    reports = re.split(r'(?=PRELIMINARY LOCAL CLIMATOLOGICAL DATA)', text)

    records = []
    for report in reports:
        # only process reports that contain a monthly summary
        if 'AVERAGE MONTHLY:' in report and 'MONTH:' in report:
            month_match = re.search(r'MONTH:\s+(\w+)', report)
            year_match  = re.search(r'YEAR:\s+(\d{4})', report)
            avg_match   = re.search(r'AVERAGE MONTHLY:\s+(-?\d+\.?\d*)', report)
            
            if month_match and year_match and avg_match:
                records.append({
                    'year': int(year_match.group(1)),
                    'month': month_match.group(1),
                    'avg_temp_f': float(avg_match.group(1))
                })

    df_monthly = pd.DataFrame(records)

    month_temp_series = df_monthly.groupby('month')['avg_temp_f'].mean()

    month_order = ['JANUARY','FEBRUARY','MARCH','APRIL','MAY','JUNE',
                   'JULY','AUGUST','SEPTEMBER','OCTOBER','NOVEMBER','DECEMBER']
    
    month_temp = month_temp_series.reindex(month_order).dropna().to_dict()

    return month_temp

month_temp = load_fbx_weather_data()

print("=" * 70)
print("month_temp returned from load_fbx_weather_data()")
print("=" * 70)
for month, temp in month_temp.items():
    print(f"  month_temp['{month:>12s}'] = {temp:>6.1f} degF")
print("=" * 70)
print(f"Type:  {type(month_temp)}")
print(f"Keys:  {list(month_temp.keys())}")
print(f"Count: {len(month_temp)} months")

print('\nwe use these monthly average inputs in the diesel simulation ...\n' 
'still need ambients tho')

# index
# SSSF
# LHV

# BACT = best available control technologies
# SCR
# NOx
# EU8
# PCV
# PM2.5
# DPF
# ULSD

month_temp returned from load_fbx_weather_data()
  month_temp['     JANUARY'] =    3.1 degF
  month_temp['    FEBRUARY'] =   -0.2 degF
  month_temp['       MARCH'] =    8.4 degF
  month_temp['       APRIL'] =   31.5 degF
  month_temp['         MAY'] =   49.4 degF
  month_temp['        JUNE'] =   61.0 degF
  month_temp['        JULY'] =   64.0 degF
  month_temp['      AUGUST'] =   58.5 degF
  month_temp['   SEPTEMBER'] =   46.6 degF
  month_temp['     OCTOBER'] =   29.8 degF
  month_temp['    NOVEMBER'] =    8.9 degF
  month_temp['    DECEMBER'] =   -2.4 degF
Type:  <class 'dict'>
Keys:  ['JANUARY', 'FEBRUARY', 'MARCH', 'APRIL', 'MAY', 'JUNE', 'JULY', 'AUGUST', 'SEPTEMBER', 'OCTOBER', 'NOVEMBER', 'DECEMBER']
Count: 12 months

we use these monthly average inputs in the diesel simulation ...
still need ambients tho


In [ ]:
# 1
'''assumptions: 

'''

In [ ]:
# 1.1
# emissions modeling&control CHECKS

# operational
load_EU8 = 0
if load_EU8 < 0.4:
    print('VIOLATION1: BACT optimization for SCR system, load >40% required to maintain efficiency \n')

exhaust_gas_temp = 0
if exhaust_gas_temp != range(270,400): # in degC
    print('VIOLATION2: exhaust gas temperature is too low. to effectively reduce NOx \n' 
    'temperatures must be within 270-400 C. temperatures below this can form ammonium sulfates that \n' 
    'destroy the catalyst\n')

intake_air_temp = 121 # in degC
 # manifold, since EU8 uses turbocharger and aftercooler, critical for modeling NOx ...
# reduction, lowering intake air temp -> reduce peak flame temp in combustion chamber

# control tech
eta_NOx = 1*100 # NOx removal efficiency
if eta_NOx != range(80,90): # in %
    print('VIOLATION3: BACT required SCR system should target a reduction of ~80-90% \n')

# molar ratio of ammonia:NOx is primary control var for SCR performance

nh3_slip = 10 # in ppmv
# constraint, make sure unreacted ammonia in exhaust < permissible levels

pcv_reintro_rate = 0.1 # 10% control
# efficiency of PCV system in reintroducing unburned fuel into cylinder, heat sink ...
# to further lower thermal NOx, reduce PM-2.5

# physical, fuel constraint
p_back = 0 # exhaust backpressure, MOST CRITICAL ...
# BACT det for EU8 excluses DPF, resulting p_back > manufacturer limits -> mechanical failure/overheating

fuel_sulfur_content = 0.0015/100 # 0.0015% by wt, 15 ppm for ULSD
fuel_ash_content = 0.38/100 # 0.38% ash BUT current BACT mandates low ash diesel
# assume exclusively low ash distallate fuel to prevent equipement fouling ...
# meet PM2.5 standards 

# regulatory&economic optimization
yr_op_hr_limit = 0 # yearly operating hr limit, for non emergency operation ...
# ex. maint, readiness testing capped at 100 hrs/yr
if yr_op_hr_limit > 100:
    print('VIOLATION4: yearly operating hour limit exceeded, must be under 100 hrs/yr \n')

emission_limit_NOx = 40 # in tons/yr
# EU8 & EU4(backup boiler) share an emission limit of 40 tons/yr

our_NOx = 0 # in g/hp-hr | check against BACT targets
if our_NOx <= 1.3:
    print('NOx compliant with BACT \n')
else:
    print('VIOLATION5: NOx NOT compliant with BACT \n')

our_PM2pt5 = 0 # in g/hp-hr
if our_PM2pt5 <= 0.32:
    print('PM-2.5 compliant with BACT \n')
else:
    print('VIOLATION6: PM-2.5 NOT compliant with BACT \n')